# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

- [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure `mlcroissant` (and plotting dependencies) are installed
!pip install mlcroissant matplotlib seaborn --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata and dataset object
dataset = mlc.Dataset(croissant_url)

# Print summary metadata info
print(f"Dataset: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published: {dataset.metadata.datePublished if hasattr(dataset.metadata, 'datePublished') else 'N/A'}")
print(f"License: {dataset.metadata.license if hasattr(dataset.metadata, 'license') else 'N/A'}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

A Croissant dataset organizes tabular data in **record sets**. Each record set has fields (columns), each with a unique `@id`.

In [ ]:
# List all record sets and their fields with `@id`s
record_sets = list(dataset.metadata.record_sets)
print("Available record sets:\n======================")
for recset in record_sets:
    print(f"Record set `@id`: {recset.id} | Name: {recset.name}")
    if hasattr(recset, 'description'):
        print(f"  Description: {recset.description}")
    print("  Fields (`@id`):")
    for field in recset.fields:
        print(f"      - {field.id} | Name: {field.name}")
    print()

### Preview a sample of records from the primary record set

Below, we preview a couple of records for the main record set (using its `@id`).

In [ ]:
# For demonstration, use the first record set (the main table). Adjust `record_set_id` as needed if there are multiple sets.
if len(record_sets) == 0:
    raise ValueError("No record sets found in dataset metadata.")
record_set_id = record_sets[0].id
print(f"Previewing first 2 records of record set '@id': {record_set_id}\n")
for i, record in enumerate(dataset.records(record_set=record_set_id)):
    print(record)
    if i >= 1:
        break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# List all available record set @ids (for future referencing)
all_record_set_ids = [rs.id for rs in record_sets]
print(f"All record sets: {all_record_set_ids}")

# Load records from all tables into pandas DataFrames, keyed by record set @id
dataframes = {}
for rs in record_sets:
    print(f"Loading record set: {rs.id}")
    df = pd.DataFrame(list(dataset.records(record_set=rs.id)))
    dataframes[rs.id] = df

print(f"\nColumns for table '{record_set_id}':")
print(dataframes[record_set_id].columns.tolist())

print("\nFirst 5 rows:")
dataframes[record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes to prepare for further analysis.

**Note:**
You must select the correct field `@id`s for filtering and grouping below. These are shown in the Data Overview and can be obtained by inspecting the DataFrame's columns.

In [ ]:
# --- CONFIGURE ---
# Update these field @ids for your own analysis (shown in overview/code above)

# Suppose the protein abundance field is called 'abundance', 'normalized_abundance' or similar (inspect the columns above).
# Let's search for a numeric field for demonstration. If not, substitute with an available numeric field/column name.
import numpy as np

df = dataframes[record_set_id]
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
if not numeric_cols:
    # Try to parse numbers if the data is in string format (often the case after CSV ingestion)
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col], errors='ignore')
        except Exception:
            continue
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

if numeric_cols:
    numeric_field = numeric_cols[0]
else:
    raise ValueError('No numeric columns found in the data.')
print(f"Using numeric field for EDA: {numeric_field}")

# Set a threshold (e.g., filter proteins with abundance > 10) - for demonstration
threshold = 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Records with {numeric_field} > {threshold}: {len(filtered_df)}")
filtered_df = filtered_df.copy()

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"First 5 rows with normalized {numeric_field}:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a categorical field if available (e.g., 'description', 'accession', etc.)
possible_group_fields = [c for c in df.columns if c not in numeric_cols and df[c].nunique() < 20 and df[c].dtype == 'object']
if possible_group_fields:
    group_field = possible_group_fields[0]
    print(f"Grouping by: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    print(grouped_df.head())
else:
    print("No suitable group field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Adjust plot code to relevant field `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True, color='skyblue')
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

# If a group field was found, show mean/norm by group
if 'group_field' in locals():
    plt.figure(figsize=(10, 4))
    group_stats = filtered_df.groupby(group_field)[f"{numeric_field}_normalized"].mean().sort_values()
    group_stats.plot(kind='bar', color='orange')
    plt.title(f"Mean Normalized {numeric_field} by {group_field}")
    plt.ylabel(f"Mean Normalized {numeric_field}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR^2 Croissant dataset using `mlcroissant`, previewed record sets and field `@id`s, read table rows into DataFrames, and performed elementary exploratory analysis including filtering, normalization, groupwise statistics, and simple data visualizations. For deeper bioinformatics or ML analyses, continue using the DataFrame objects and field `@id`s referenced above.

Remember: Croissant's explicit linking of schema entities by `@id` ensures clarity and reproducibility for data selection, processing, and downstream ML pipelines.